# 🍽️ Step 1 — Data Pipeline
### Multimodal Food Recognition & Nutrition Assistant

**Goal:** Merge all datasets into one clean master nutrition table.

**Join Chain:**
```
food_category ──► food ──► food_nutrient ──► nutrient
     (id)        (fdc_id)   (nutrient_id)      (id)
```

**Output:** `master_nutrition.csv` — used by all future steps

## 📦 1.1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ⚙️ 1.2 — Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries ready')

## 📁 1.3 — Configure File Paths

In [ ]:
# ─── SET YOUR BASE FOLDER PATH HERE ──────────────────────────────────────
# This is the folder in your Google Drive where all CSV files are stored.
BASE = '/content/drive/MyDrive/'  # Change this if files are in a subfolder
# ─────────────────────────────────────────────────────────────────────────

PATHS = {
    'food'          : BASE + 'food.csv',
    'nutrient'      : BASE + 'nutrient.csv',
    'food_nutrient' : BASE + 'food_nutrient.csv',
    'food_category' : BASE + 'food_category.csv',
    'recipes'       : BASE + 'full_dataset.csv',
}

# Verify all files exist before loading
all_found = True
for name, path in PATHS.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ NOT FOUND'
    print(f'{status}  {name:20s} → {path}')
    if not exists:
        all_found = False

if not all_found:
    print('\n⚠️  Some files are missing. Update BASE path above and re-run.')
else:
    print('\n✅ All files found. Proceed to next cell.')

## 📥 1.4 — Load Raw Datasets

In [ ]:
print('Loading datasets...\n')

food_df          = pd.read_csv(PATHS['food'])
nutrient_df      = pd.read_csv(PATHS['nutrient'])
food_nutrient_df = pd.read_csv(PATHS['food_nutrient'])
food_category_df = pd.read_csv(PATHS['food_category'])
recipes_df       = pd.read_csv(PATHS['recipes'])

# Keep only useful recipe columns
recipes_df = recipes_df[['title', 'ingredients']]

print(f'food.csv          → {food_df.shape[0]:,} rows, {food_df.shape[1]} cols')
print(f'nutrient.csv      → {nutrient_df.shape[0]:,} rows, {nutrient_df.shape[1]} cols')
print(f'food_nutrient.csv → {food_nutrient_df.shape[0]:,} rows, {food_nutrient_df.shape[1]} cols')
print(f'food_category.csv → {food_category_df.shape[0]:,} rows, {food_category_df.shape[1]} cols')
print(f'recipes           → {recipes_df.shape[0]:,} rows, {recipes_df.shape[1]} cols')

## 🔍 1.5 — Quick Preview of Each Dataset

In [ ]:
print('── food_category ──')
display(food_category_df.head(3))

print('── food ──')
display(food_df.head(3))

print('── nutrient ──')
display(nutrient_df.head(3))

print('── food_nutrient ──')
display(food_nutrient_df.head(3))

print('── recipes ──')
display(recipes_df.head(3))

## 🧹 1.6 — Clean Each Dataset

In [ ]:
# ── Clean food_category ──────────────────────────────────────────────────
food_category_df = food_category_df[['id', 'description']].rename(
    columns={'id': 'food_category_id', 'description': 'category_name'}
)

# ── Clean food ───────────────────────────────────────────────────────────
food_df = food_df[['fdc_id', 'description', 'food_category_id']].rename(
    columns={'description': 'food_name'}
)
# Standardise food name: lowercase, strip whitespace
food_df['food_name_clean'] = food_df['food_name'].str.lower().str.strip()

# ── Clean nutrient ───────────────────────────────────────────────────────
nutrient_df = nutrient_df[['id', 'name', 'unit_name']].rename(
    columns={'id': 'nutrient_id', 'name': 'nutrient_name', 'unit_name': 'unit'}
)

# ── Clean food_nutrient ──────────────────────────────────────────────────
food_nutrient_df = food_nutrient_df[['fdc_id', 'nutrient_id', 'amount']].dropna(subset=['amount'])
food_nutrient_df['amount'] = pd.to_numeric(food_nutrient_df['amount'], errors='coerce')
food_nutrient_df = food_nutrient_df.dropna(subset=['amount'])
food_nutrient_df = food_nutrient_df[food_nutrient_df['amount'] >= 0]  # Remove negatives

print('✅ All datasets cleaned')

## 🔗 1.7 — Merge into Master Nutrition Table

In [ ]:
# Step 1: food + category
master = food_df.merge(food_category_df, on='food_category_id', how='left')

# Step 2: + food_nutrient values
master = master.merge(food_nutrient_df, on='fdc_id', how='inner')

# Step 3: + nutrient names & units
master = master.merge(nutrient_df, on='nutrient_id', how='inner')

# Keep only the columns we need
master = master[['fdc_id', 'food_name', 'food_name_clean', 'category_name',
                 'nutrient_name', 'amount', 'unit']]

print(f'✅ Master table shape: {master.shape[0]:,} rows × {master.shape[1]} cols')
display(master.head(10))

## 🎯 1.8 — Filter to Key Nutrients Only

In [ ]:
# These are the 14 nutrients most relevant for health & nutrition analysis
KEY_NUTRIENTS = [
    'Energy',
    'Protein',
    'Total lipid (fat)',
    'Carbohydrate, by difference',
    'Fiber, total dietary',
    'Sugars, total including NLEA',
    'Sodium, Na',
    'Calcium, Ca',
    'Iron, Fe',
    'Potassium, K',
    'Vitamin C, total ascorbic acid',
    'Vitamin A, RAE',
    'Cholesterol',
    'Fatty acids, total saturated'
]

master_filtered = master[master['nutrient_name'].isin(KEY_NUTRIENTS)].copy()

print(f'Full master rows    : {master.shape[0]:,}')
print(f'Filtered master rows: {master_filtered.shape[0]:,}')
print(f'Unique foods        : {master_filtered["fdc_id"].nunique():,}')
print(f'Nutrients tracked   : {master_filtered["nutrient_name"].nunique()}')
print(f'\nNutrients in filtered table:')
print(master_filtered['nutrient_name'].unique())

## 🔄 1.9 — Pivot to Wide Format (One Row Per Food)

In [ ]:
# Wide format: each nutrient becomes its own column
# This makes lookup fast and easy for the chatbot layer

nutrition_wide = master_filtered.pivot_table(
    index=['fdc_id', 'food_name', 'food_name_clean', 'category_name'],
    columns='nutrient_name',
    values='amount',
    aggfunc='first'
).reset_index()

# Flatten column names
nutrition_wide.columns.name = None

# Rename for clarity
nutrition_wide = nutrition_wide.rename(columns={
    'Energy'                        : 'calories_kcal',
    'Protein'                       : 'protein_g',
    'Total lipid (fat)'             : 'fat_g',
    'Carbohydrate, by difference'   : 'carbs_g',
    'Fiber, total dietary'          : 'fiber_g',
    'Sugars, total including NLEA'  : 'sugar_g',
    'Sodium, Na'                    : 'sodium_mg',
    'Calcium, Ca'                   : 'calcium_mg',
    'Iron, Fe'                      : 'iron_mg',
    'Potassium, K'                  : 'potassium_mg',
    'Vitamin C, total ascorbic acid': 'vitamin_c_mg',
    'Vitamin A, RAE'                : 'vitamin_a_mcg',
    'Cholesterol'                   : 'cholesterol_mg',
    'Fatty acids, total saturated'  : 'saturated_fat_g'
})

print(f'✅ Wide table: {nutrition_wide.shape[0]:,} foods × {nutrition_wide.shape[1]} columns')
display(nutrition_wide.head(5))

## 🍳 1.10 — Clean the Recipes Dataset

In [ ]:
import ast

def parse_ingredients(raw):
    """Safely parse ingredients from string representation of a list."""
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed]
    except Exception:
        pass
    # Fallback: treat as plain text
    return [raw.strip()] if isinstance(raw, str) else []

recipes_df = recipes_df.dropna(subset=['title', 'ingredients'])
recipes_df['ingredients_list'] = recipes_df['ingredients'].apply(parse_ingredients)
recipes_df['ingredients_text'] = recipes_df['ingredients_list'].apply(lambda x: ', '.join(x))
recipes_df['title_clean']      = recipes_df['title'].str.lower().str.strip()

# Combined text field for embedding/search
recipes_df['full_text'] = recipes_df['title'] + '. Ingredients: ' + recipes_df['ingredients_text']

print(f'✅ Recipes cleaned: {len(recipes_df):,} recipes')
print(f'   Sample entry:')
print(f'   Title      : {recipes_df.iloc[0]["title"]}')
print(f'   Ingredients: {recipes_df.iloc[0]["ingredients_text"][:120]}...')

## 🔎 1.11 — Build a Fast Food Lookup Function

In [ ]:
def lookup_nutrition(food_query: str, top_n: int = 5) -> pd.DataFrame:
    """
    Search the nutrition database by food name.
    Returns top_n matching foods with their full nutrition profile.
    
    Args:
        food_query : plain text food name, e.g. 'chicken breast'
        top_n      : how many matches to return
    Returns:
        DataFrame of matching foods with nutrition columns
    """
    query = food_query.lower().strip()
    
    # Exact match first
    exact = nutrition_wide[nutrition_wide['food_name_clean'] == query]
    if len(exact) >= 1:
        return exact.head(top_n)
    
    # Partial / keyword match
    keywords = query.split()
    mask = nutrition_wide['food_name_clean'].str.contains(keywords[0], na=False)
    for kw in keywords[1:]:
        mask &= nutrition_wide['food_name_clean'].str.contains(kw, na=False)
    
    results = nutrition_wide[mask]
    
    if len(results) == 0:
        # Broad fallback: any keyword
        mask = nutrition_wide['food_name_clean'].str.contains(keywords[0], na=False)
        results = nutrition_wide[mask]
    
    return results.head(top_n)


def nutrition_summary(food_query: str) -> str:
    """
    Returns a human-readable nutrition summary string for a given food.
    Used as context input to the LLM chatbot in Step 4.
    """
    results = lookup_nutrition(food_query, top_n=1)
    
    if results.empty:
        return f'No nutrition data found for "{food_query}" in the USDA database.'
    
    row = results.iloc[0]
    
    def val(col):
        v = row.get(col, None)
        return f'{v:.1f}' if pd.notna(v) else 'N/A'
    
    summary = f"""
NUTRITION INFO (per 100g): {row['food_name']}
Category   : {row.get('category_name', 'Unknown')}
─────────────────────────────
Calories   : {val('calories_kcal')} kcal
Protein    : {val('protein_g')} g
Fat        : {val('fat_g')} g
  Saturated: {val('saturated_fat_g')} g
Carbs      : {val('carbs_g')} g
  Fiber    : {val('fiber_g')} g
  Sugar    : {val('sugar_g')} g
Sodium     : {val('sodium_mg')} mg
Cholesterol: {val('cholesterol_mg')} mg
Calcium    : {val('calcium_mg')} mg
Iron       : {val('iron_mg')} mg
Potassium  : {val('potassium_mg')} mg
Vitamin C  : {val('vitamin_c_mg')} mg
Vitamin A  : {val('vitamin_a_mcg')} mcg
"""
    return summary.strip()


# ── Test it ──────────────────────────────────────────────────────────────
print(nutrition_summary('chicken breast'))
print('\n── Top 5 matches for "apple" ──')
display(lookup_nutrition('apple')[['food_name', 'category_name', 'calories_kcal', 'protein_g', 'carbs_g']])

## 💾 1.12 — Save All Outputs

In [ ]:
OUTPUT_DIR = BASE + 'nutrition_assistant_outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save master wide table (primary lookup for all future steps)
nutrition_wide.to_csv(OUTPUT_DIR + 'master_nutrition.csv', index=False)

# Save cleaned long-format table (useful for analytics)
master_filtered.to_csv(OUTPUT_DIR + 'master_nutrition_long.csv', index=False)

# Save cleaned recipes
recipes_df[['title', 'title_clean', 'ingredients_text', 'full_text']].to_csv(
    OUTPUT_DIR + 'recipes_clean.csv', index=False
)

print('✅ Files saved to:', OUTPUT_DIR)
print('   master_nutrition.csv      ← used by Step 2, 3, 4')
print('   master_nutrition_long.csv ← useful for analytics')
print('   recipes_clean.csv         ← used by Step 3 (RAG)')

## ✅ Step 1 Complete!

You now have:

| File | Purpose |
|---|---|
| `master_nutrition.csv` | One row per food, one column per nutrient |
| `master_nutrition_long.csv` | Long format, good for charts & analysis |
| `recipes_clean.csv` | Clean recipes ready for RAG embedding |

**Functions built:**
- `lookup_nutrition(food_query)` — search foods by name, returns DataFrame
- `nutrition_summary(food_query)` — returns a text summary for the LLM chatbot

---
**➡️ Next: Step 2 — Food Image Recognition (CLIP Model)**